# OpenMantra Pipeline Evaluation

Evaluates the full manga translation pipeline on OpenMantra dataset:
- **OCR Accuracy**: CER on matched bubbles
- **Translation Quality**: BLEU, chrF++, BERTScore, COMET on matched bubbles  
- **Reading Order**: Kendall's Tau correlation
- **Coverage**: GT text coverage, bubble utilization

Uses containment-based matching: GT text bbox must be ≥80% inside predicted bubble.


In [ ]:
import os
import sys

ENDSWITH = 'OpenMantra'
NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDSWITH):
    raise ValueError(f"Expected dir ending with {ENDSWITH}, got {NOTEBOOK_DIR}")

BASE_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..', '..')
sys.path.insert(0, BASE_DIR)
print(f"Base dir: {BASE_DIR}")


In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional, Union
from tqdm.auto import tqdm
from scipy.stats import kendalltau
import torch
from torch.utils.data import Dataset
from torchmetrics.text import CharErrorRate, WordErrorRate, BLEUScore, SacreBLEUScore, CHRFScore
from torchmetrics.text.bert import BERTScore

try:
    from comet import download_model, load_from_checkpoint
    COMET_AVAILABLE = True
except ImportError:
    COMET_AVAILABLE = False
    print("Warning: COMET not available. Install with: pip install unbabel-comet")

from src.pipeline.SegmentationModels.YoloSeg import YoloBubbleSeg, YoloPanelSeg
from src.pipeline.SegmentationModels.BubbleSegmenterWithSplit import BubbleSegmenterWithSplit
from src.pipeline.SegmentationModels.BubbleSegmentationWithOrdering import BubbleSegmentationWithOrdering
from src.pipeline.OCRModels.MangaOCRModel import MangaOCRModel
from src.pipeline.TranslationModels.ElanMtJaEnTranslator import ElanMtJaEnTranslator
from src.pipeline.TranslationModels.ElanMtJaEnBatchTranslator import ElanMtJaEnBatchTranslator
from src.pipeline.TranslationModels.ContextAwareLLMTranslator import ContextAwareLLMTranslator
from src.pipeline.Utils.MangaTypesetter import MangaTypesetter
from src.pipeline.Utils.MangaPipeline import MangaPipeline

print("Imports successful!")


In [ ]:
# Configuration
DEVICE = "auto"  # Change to "cuda" or "cpu" as needed
OPENMANTRA_ROOT = os.path.join(BASE_DIR, "data/open-mantra-dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "output/pipeline/Evaluators/Pipeline/OpenMantra")
CONTAINMENT_THRESHOLD = 0.8  # GT text must be 80% inside bubble

from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUTPUT_DIR, f"run_{RUN_ID}")

PLOTS_DIR = os.path.join(RUN_DIR, "plots")
VIS_DIR = os.path.join(RUN_DIR, "visualizations")
TRANS_DIR = os.path.join(RUN_DIR, "translations")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(TRANS_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"OpenMantra root: {OPENMANTRA_ROOT}")
print(f"Run dir: {RUN_DIR}")


## 1. Load OpenMantra Dataset

In [ ]:
from helper import OpenMantraDataset

dataset = OpenMantraDataset(root_dir=OPENMANTRA_ROOT)

print(f"Total pages: {len(dataset)}")
print(f"Books: {dataset.get_book_titles()}")


## 2. Create Pipeline


In [ ]:
panel_detector = YoloPanelSeg(variant="v8n", device=DEVICE, verbose=False)

bubble_detector = BubbleSegmenterWithSplit(
    variant="v8s",
    device=DEVICE,
    verbose=False,
    plot=False
)

ordered_segmenter = BubbleSegmentationWithOrdering(
    bubble_detector=bubble_detector,
    panel_detector=panel_detector,
    num_panel_rows=4,
    right_to_left=True,
    verbose=False,
    plot=False
)

ocr_model = MangaOCRModel(verbose=False)
translator = ElanMtJaEnTranslator(elan_model="tiny", device=DEVICE, verbose=False)
# translator = ContextAwareLLMTranslator(model_name='TheBlindMaster/Qwen2.5-0.5B-Instruct-emergent-finetune-train_full_context-all-full-r64-e1', context_window=1, verbose=True, batch_size=1)

typesetter = MangaTypesetter()

pipeline = MangaPipeline(
    segmenter=ordered_segmenter,
    ocr_model=ocr_model,
    translator=translator,
    typesetter=typesetter,
    verbose=False
)

print("Pipeline created!")


## 3. Evaluation Functions


In [ ]:
from helper import match_bubbles_to_gt_texts, evaluate_ocr_page, evaluate_translation_page, evaluate_ordering_page, evaluate_page, visualize_matching, save_translation_output, create_summary_plots, compute_final_metrics

## 4. Run Evaluation


In [ ]:
# Test on single image first
print("Testing on single image...")

pipeline.load_models()

# Get first sample (skip cover pages)
TEST_IDX = 5
pil_image, image_info, gt_texts = dataset[TEST_IDX]
image = np.array(pil_image)

print(f"Image: {image_info['book_title']} page {image_info['page_index']}")
print(f"GT texts: {len(gt_texts)}")

# Run evaluation
result = evaluate_page(
    pipeline, image, gt_texts, 
    containment_threshold=CONTAINMENT_THRESHOLD,
    conf_threshold=0.5
)

print(f"\nResults:")
print(f"  OCR samples: {len(result['ocr_predictions'])}")
print(f"  Trans samples: {len(result['trans_predictions'])}")
print(f"  Coverage: {result['coverage']}")
print(f"  Ordering: {result['ordering']}")

# Save test visualization
if result['pred_bboxes']:
    test_vis_path = os.path.join(RUN_DIR, "test_visualization.png")
    visualize_matching(
        image, 
        result['pred_bboxes'], 
        result['valid_gt_texts'],
        result['matches'],
        save_path=test_vis_path
    )
    print(f"\nTest visualization saved to: {test_vis_path}")


In [ ]:
# Show OCR and Translation comparisons
print("OCR Comparison:")
for i, (pred, exp) in enumerate(zip(result['ocr_predictions'][:5], result['ocr_expected'][:5])):
    print(f"  [{i}] Pred: {pred}")
    print(f"      GT:   {exp}")
    print()

print("\nTranslation Comparison:")
for i, (pred, exp) in enumerate(zip(result['trans_predictions'][:5], result['trans_expected'][:5])):
    print(f"  [{i}] Pred: {pred}")
    print(f"      GT:   {exp}")
    print()


## 5. Full Dataset Evaluation


In [ ]:
# Run full evaluation with visualization saving
all_results = []
VIS_SAMPLE_INTERVAL = 1  # Save visualization every N pages

print(f"Evaluating {len(dataset)} pages...")
print(f"Saving visualizations every {VIS_SAMPLE_INTERVAL} pages to {VIS_DIR}")

for idx in tqdm(range(len(dataset))):
    pil_image, image_info, gt_texts = dataset[idx]
    image = np.array(pil_image)
    
    result = evaluate_page(
        pipeline, image, gt_texts,
        containment_threshold=CONTAINMENT_THRESHOLD,
        conf_threshold=0.5
    )
    
    result['image_info'] = image_info
    all_results.append(result)
    
    # Save visualization for sample pages
    if idx % VIS_SAMPLE_INTERVAL == 0 and result['pred_bboxes']:
        vis_path = os.path.join(VIS_DIR, f"page_{idx:04d}_{image_info['book_title']}.png")
        visualize_matching(
            image, 
            result['pred_bboxes'], 
            result['valid_gt_texts'],
            result['matches'],
            save_path=vis_path
        )
    
    # Save translation output for every page
    trans_path = os.path.join(TRANS_DIR, f"page_{idx:04d}_{image_info['book_title']}.json")
    save_translation_output(
        result['ocr_predictions'],
        result['ocr_expected'],
        result['trans_predictions'],
        result['trans_expected'],
        result.get('trans_sources', []),
        image_info,
        trans_path
    )

print("Evaluation complete!")


In [ ]:
# Compute final metrics
metrics = compute_final_metrics(all_results, device=DEVICE)

print("=" * 60)
print("PIPELINE EVALUATION RESULTS")
print("=" * 60)

print("\n🎯 Detection Statistics:")
print(f"   Total GT Bubbles:   {metrics.get('detection_total_gt_texts', 'N/A')}")
print(f"   Matched Pairs:      {metrics.get('detection_matched_pairs_count', 0)} ({metrics.get('detection_matched_pairs_pct', 0):.2f}%)")
print(f"   Missed Detections:  {metrics.get('detection_missed_count', 0)} ({metrics.get('detection_missed_pct', 0):.2f}%)")
print(f"   False Positives:    {metrics.get('detection_false_positive_count', 0)} ({metrics.get('detection_false_positive_pct', 0):.2f}%)")

print("\n📝 OCR Metrics (Matched Bubbles):")
print(f"   CER: {metrics.get('ocr_cer', 1.0):.4f}")

print("\n🌐 Translation Metrics (OCR Input):")
print(f"   CER:           {metrics.get('trans_cer', 1.0):.4f}")
print(f"   BLEU:          {metrics.get('trans_bleu', 0.0):.4f}")
print(f"   SacreBLEU:     {metrics.get('trans_sacrebleu', 0.0):.4f}")
print(f"   chrF:          {metrics.get('trans_chrf', 0.0):.4f}")
print(f"   chrF++:        {metrics.get('trans_chrf_pp', 0.0):.4f}")
print(f"   BERTScore F1:  {metrics.get('trans_bertscore_f1', 0.0):.4f}")
print(f"   COMET:         {metrics.get('trans_comet', 0.0):.4f}")

print("\n✨ Translation Metrics (GT Source / Simulated Perfect OCR):")
print(f"   CER:           {metrics.get('trans_gt_source_cer', 1.0):.4f}")
print(f"   BLEU:          {metrics.get('trans_gt_source_bleu', 0.0):.4f}")
print(f"   SacreBLEU:     {metrics.get('trans_gt_source_sacrebleu', 0.0):.4f}")
print(f"   chrF:          {metrics.get('trans_gt_source_chrf', 0.0):.4f}")
print(f"   chrF++:        {metrics.get('trans_gt_source_chrf_pp', 0.0):.4f}")
print(f"   BERTScore F1:  {metrics.get('trans_gt_source_bertscore_f1', 0.0):.4f}")
print(f"   COMET:         {metrics.get('trans_gt_source_comet', 0.0):.4f}")

print("\n📖 Ordering Metrics:")
print(f"   Kendall's Tau:     {metrics.get('ordering_kendall_tau', 0.0):.4f}")
print(f"   Exact Match Rate:  {metrics.get('ordering_exact_match_rate', 0.0):.4f}")

print("\n📊 Coverage Metrics:")
print(f"   GT Coverage:        {metrics.get('gt_coverage_mean', 0.0):.4f}")
print(f"   Bubble Utilization: {metrics.get('bubble_utilization_mean', 0.0):.4f}")

print("\n📈 Counts:")
print(f"   Pages:         {metrics.get('num_pages', 0)}")
print(f"   OCR Samples:   {metrics.get('num_ocr_samples', 0)}")
print(f"   Trans Samples: {metrics.get('num_trans_samples', 0)}")

print("=" * 60)

## 6. Save Results


In [ ]:
# Save metrics
metrics_path = os.path.join(RUN_DIR, "metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

# Save detailed results
detailed_results = []
for r in all_results:
    detailed_results.append({
        'image_info': r['image_info'],
        'coverage': r['coverage'],
        'ordering': {k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
                     for k, v in r['ordering'].items()},
        'num_ocr_samples': len(r['ocr_predictions']),
        'num_trans_samples': len(r['trans_predictions'])
    })

results_path = os.path.join(RUN_DIR, "detailed_results.json")
with open(results_path, 'w') as f:
    json.dump(detailed_results, f, indent=2, default=str)
print(f"Detailed results saved to: {results_path}")

# Save all OCR and translation outputs combined (Including GT Source experiment)
all_outputs = {
    'ocr': {'predictions': [], 'expected': []},
    'translation': {'predictions': [], 'expected': [], 'sources': []},
    'translation_gt_source': {'predictions': [], 'expected': []}
}
for r in all_results:
    all_outputs['ocr']['predictions'].extend(r['ocr_predictions'])
    all_outputs['ocr']['expected'].extend(r['ocr_expected'])
    all_outputs['translation']['predictions'].extend(r['trans_predictions'])
    all_outputs['translation']['expected'].extend(r['trans_expected'])
    all_outputs['translation']['sources'].extend(r.get('trans_sources', []))
    
    # Save GT Source experiment outputs
    if 'trans_predictions_gt_source' in r:
        all_outputs['translation_gt_source']['predictions'].extend(r['trans_predictions_gt_source'])
        all_outputs['translation_gt_source']['expected'].extend(r.get('trans_expected_gt_source', []))

outputs_path = os.path.join(RUN_DIR, "all_outputs.json")
with open(outputs_path, 'w', encoding='utf-8') as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)
print(f"All outputs saved to: {outputs_path}")

# Create summary plots
create_summary_plots(all_results, PLOTS_DIR)

# Save config
config = {
    'device': DEVICE,
    'containment_threshold': CONTAINMENT_THRESHOLD,
    'num_pages': len(dataset),
    'run_id': RUN_ID,
    'openmantra_root': OPENMANTRA_ROOT
}
config_path = os.path.join(RUN_DIR, "config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved to: {config_path}")

print(f"\n✅ All results saved to: {RUN_DIR}")

In [ ]:
# Cleanup
pipeline.unload_models()
print("Models unloaded.")


## 7. Analysis

### Per-Book Breakdown


In [ ]:
# Per-book analysis
from collections import defaultdict

book_results = defaultdict(list)
for r in all_results:
    book_title = r['image_info']['book_title']
    book_results[book_title].append(r)

print("Per-Book Coverage:")
print("-" * 50)
for book_title, results in book_results.items():
    coverages = [r['coverage']['gt_coverage'] for r in results]
    taus = [r['ordering']['kendall_tau'] for r in results if r['ordering']['num_matched_bubbles'] >= 2]
    
    print(f"{book_title}:")
    print(f"   Pages: {len(results)}")
    print(f"   Avg GT Coverage: {np.mean(coverages):.4f}")
    print(f"   Avg Kendall Tau: {np.mean(taus):.4f}" if taus else "   Avg Kendall Tau: N/A")
    print()


## Summary

### Metrics Interpretation

| Metric | Description | Good Value |
|--------|-------------|------------|
| **OCR CER** | Character Error Rate | Lower is better (<0.1) |
| **Trans BLEU** | Translation quality (n-gram overlap) | Higher is better (>0.3) |
| **Trans chrF++** | Translation quality (char-level) | Higher is better (>0.4) |
| **Trans BERTScore F1** | Semantic similarity (BERT embeddings) | Higher is better (>0.8) |
| **Trans COMET** | Translation quality (neural, uses source) | Higher is better (>0.5) |
| **Kendall's Tau** | Reading order correlation | Higher is better (>0.8) |
| **Exact Match Rate** | % pages with perfect order | Higher is better |
| **GT Coverage** | % of GT texts matched to bubbles | Higher is better (>0.8) |
| **Bubble Utilization** | % of bubbles containing GT text | Higher is better |

### Notes
- Metrics are computed only on **matched** bubble-text pairs
- **BERTScore** measures semantic similarity but can be forgiving of grammar/word order issues
- **COMET** uses source text context and correlates well with human judgments
- Low GT Coverage may indicate:
  - Bubble detection misses
  - Containment threshold too strict
  - Text outside bubbles in dataset
- Low Kendall's Tau may indicate ordering algorithm issues
